# ITELv5 · Notebook 07 · Prueba del Modelo
Infiere sobre un audio nuevo usando el pipeline de stacking:
`wav2vec2-large-xlsr-53` → embedding 6144-dim → 5 MLP base → LogReg meta-modelo

In [26]:
import warnings, sys
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import joblib
from pathlib import Path
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
warnings.filterwarnings('ignore')

_anchor = 'ITELv5_01_CONFIGURACION.ipynb'
_cwd = Path('.').resolve()
if   (_cwd / _anchor).exists():                             BASE_DIR = _cwd.parent
elif (_cwd / 'notebooks' / _anchor).exists():               BASE_DIR = _cwd
elif (_cwd / 'research' / 'notebooks' / _anchor).exists():  BASE_DIR = _cwd / 'research'
else:                                                        BASE_DIR = _cwd.parent

class ITELConfig:
    BASE_DIR       = BASE_DIR
    PROCESADOS_DIR = BASE_DIR / 'processed_data'
    MODELS_DIR     = BASE_DIR.parent / 'assets' / 'models'
    SEED           = 42
    EMB_DIM        = 6144

MODEL_CACHE = ITELConfig.MODELS_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {device}')
print(f'MODELS_DIR : {ITELConfig.MODELS_DIR}')


Device     : cpu
MODELS_DIR : C:\Users\USER\Documents\GitHub\ITEL\assets\models


In [27]:
class MLP(nn.Module):
    def __init__(self, in_dim=6144, hidden_dims=[512,256,128],
                 num_classes=2, dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

print('MLP definido (in_dim=6144)')


MLP definido (in_dim=6144)


In [28]:
# ── Modelo de embeddings: wav2vec2-large-xlsr-53 (1024 dims) ─────────────
MODEL_ID = 'facebook/wav2vec2-large-xlsr-53'
print(f'Cargando {MODEL_ID} ...')

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    MODEL_ID, cache_dir=str(MODEL_CACHE)
)
xlsr_model = Wav2Vec2Model.from_pretrained(
    MODEL_ID, cache_dir=str(MODEL_CACHE)
).to(device)
xlsr_model.eval()
print('wav2vec2-large-xlsr-53 cargado (dim=1024)')


Cargando facebook/wav2vec2-large-xlsr-53 ...


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


wav2vec2-large-xlsr-53 cargado (dim=1024)


In [29]:
NUM_MODELS  = 5
NUM_CLASSES = 2   # Healthy / Patients

scaler     = joblib.load(ITELConfig.MODELS_DIR / 'final_mlp_scaler.joblib')
meta_model = joblib.load(ITELConfig.MODELS_DIR / 'final_meta_model_v5.joblib')
le         = joblib.load(ITELConfig.MODELS_DIR / 'label_encoder_v5.joblib')

models = []
for i in range(NUM_MODELS):
    m = MLP(in_dim=ITELConfig.EMB_DIM, num_classes=NUM_CLASSES).to(device)
    m.load_state_dict(torch.load(
        ITELConfig.MODELS_DIR / f'base_mlp_{i+1}_v5.pth',
        map_location=device
    ))
    m.eval()
    models.append(m)

print(f'{len(models)} modelos base cargados.')
print(f'Clases: {le.classes_}')


5 modelos base cargados.
Clases: ['Healthy' 'Patients']


In [30]:
def extract_embedding(audio_path: Path) -> np.ndarray:
    """
    Extrae embedding de 6144 dims usando wav2vec2-large-xlsr-53.
    Igual que NB05: mean, std, first_25%, last_25%, delta, slope → concat.
    """
    y, sr = torchaudio.load(str(audio_path))
    if y.shape[0] > 1:
        y = y.mean(dim=0, keepdim=True)
    if sr != 16000:
        y = torchaudio.transforms.Resample(sr, 16000)(y)
    y = y.squeeze(0).numpy()

    inputs = feature_extractor(
        y, sampling_rate=16000, return_tensors='pt', padding=False
    )
    input_values = inputs.input_values.to(device)

    with torch.inference_mode():
        outputs = xlsr_model(input_values)
    features = outputs.last_hidden_state.squeeze(0).cpu()  # (T, 1024)

    T = features.shape[0]
    mean_emb  = features.mean(dim=0)                         # (1024,)
    std_emb   = features.std(dim=0)                          # (1024,)
    q1 = max(1, T // 4)
    first_25  = features[:q1].mean(dim=0)                   # (1024,)
    last_25   = features[-q1:].mean(dim=0)                  # (1024,)

    # Delta (first-order difference mean)
    if T > 1:
        delta = (features[1:] - features[:-1]).mean(dim=0)  # (1024,)
    else:
        delta = torch.zeros(1024)

    # Slope via linear regression por dimension
    t = torch.linspace(0, 1, T).unsqueeze(1)                # (T,1)
    t_mean = t.mean()
    f_mean = features.mean(dim=0, keepdim=True)             # (1,1024)
    num    = ((t - t_mean) * (features - f_mean)).sum(dim=0)
    den    = ((t - t_mean) ** 2).sum()
    slope_emb = torch.where(
        features.var(dim=0) < 1e-10,
        torch.zeros(1024),
        num / (den + 1e-10)
    )                                                        # (1024,)

    emb = torch.cat([mean_emb, std_emb, first_25, last_25, delta, slope_emb])
    return emb.numpy()  # (6144,)

print('Funcion extract_embedding definida (salida: 6144 dims)')


Funcion extract_embedding definida (salida: 6144 dims)


In [31]:
def predict_audio(audio_path):
    audio_path = Path(audio_path)
    print(f'Procesando: {audio_path.name}...')

    # 1. Embedding
    emb = extract_embedding(audio_path)          # (6144,)
    emb_s = scaler.transform(emb.reshape(1, -1)) # (1, 6144)
    emb_tensor = torch.tensor(emb_s, dtype=torch.float32).to(device)

    # 2. Probabilidades de cada base MLP
    probs_list = []
    with torch.inference_mode():
        for m in models:
            logits = m(emb_tensor)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            probs_list.append(probs)

    # 3. Meta-modelo
    X_stack    = np.hstack(probs_list)               # (1, 10)
    pred_idx   = meta_model.predict(X_stack)[0]
    pred_proba = meta_model.predict_proba(X_stack)[0]

    label = le.inverse_transform([pred_idx])[0]
    conf  = pred_proba[pred_idx]

    print(f'  Prediccion : {label}')
    print(f'  Confianza  : {conf:.1%}')
    for i, cls in enumerate(le.classes_):
        print(f'    {cls}: {pred_proba[i]:.1%}')
    return label, conf

print('Funcion predict_audio lista.')
print('Uso: predict_audio("ruta/al/audio.wav")')


Funcion predict_audio lista.
Uso: predict_audio("ruta/al/audio.wav")


In [32]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import tempfile, os

upload_widget = widgets.FileUpload(
    accept='.wav,.mp3',
    multiple=False,
    description='Subir audio'
)
output = widgets.Output()

def on_upload(change):
    with output:
        clear_output()
        if not upload_widget.value:
            return
        uploaded = list(upload_widget.value.values())[0]
        fname    = list(upload_widget.value.keys())[0]
        content  = uploaded['content']
        suffix   = Path(fname).suffix
        fd, tmp_path = tempfile.mkstemp(suffix=suffix)
        os.close(fd)
        with open(tmp_path, 'wb') as f:
            f.write(bytes(content))
        print(f'Archivo recibido: {fname}')
        try:
            predict_audio(tmp_path)
        except Exception as exc:
            import traceback
            print(f'Error al procesar el archivo: {exc}')
            traceback.print_exc()
        finally:
            try:
                os.unlink(tmp_path)
            except Exception:
                pass

upload_widget.observe(on_upload, names='value')
display(upload_widget, output)


FileUpload(value=(), accept='.wav,.mp3', description='Subir audio')

Output()